<!-- SPDX-License-Identifier: Apache-2.0 -->
<!-- Copyright (c) 2025 FlyDSL Project Contributors -->

# Hello, FlyDSL

**FlyDSL** is a Python DSL and MLIR compiler stack for writing high-performance
AMD GPU kernels. You write ordinary-looking Python; FlyDSL *traces* it into the
`fly` / `fly_rocdl` MLIR dialects, lowers that through ROCDL/LLVM, and emits a
HSACO binary that runs on the GPU.

This is notebook **0 of an onboarding series** that builds up the
`flydsl.expr` foundation one idea at a time:

| # | Notebook | Topic |
|---|----------|-------|
| 00 | *this one* | the mental model: `@kernel` / `@jit`, and how to read the IR |
| 01 | `01_numeric_types` | the scalar type system (ints, floats, bf16, fp8) |
| 02 | `02_struct` | aggregate value types with `@fx.struct` |
| 03 | `03_universal_ops` | target-agnostic `Universal*` atoms + a vector-add capstone |

Layout algebra (`make_layout`, `logical_divide`, tiled copy, MMA) is intentionally
**not** covered yet — it gets its own series once these primitives are familiar.

**Prerequisites:** a built/installed `flydsl`, a ROCm GPU, and `wurlitzer`
(`pip install wurlitzer`) so the notebook can show GPU `printf` output — see below.

## 1. Setup & sanity check

Two imports cover almost everything:

- `flydsl.compiler as flyc` — the compiler entry points (`@flyc.kernel`, `@flyc.jit`, `flyc.from_dlpack`).
- `flydsl.expr as fx` — the DSL surface you write *inside* a kernel (types, ops, atoms, `printf`).

If the import below fails, FlyDSL isn't on your path yet — build it and
`pip install -e .` (see the project README), then restart the kernel.

In [ ]:
import torch

import flydsl.compiler as flyc
import flydsl.expr as fx
from flydsl.runtime.device import get_rocm_arch

print("torch sees GPU:", torch.cuda.is_available())
print("ROCm arch     :", get_rocm_arch())

**A note on seeing GPU output.** `fx.printf` runs on the device and writes to the
process's stdout, which Jupyter does not capture on its own. The tiny helper below
runs a launcher and routes that output back into the notebook (via `wurlitzer`).
We'll use `show_gpu_output(...)` throughout the series. Outside Jupyter — running a
plain `.py` — you don't need any of this; `printf` just goes to your terminal.

In [ ]:
from wurlitzer import pipes


def show_gpu_output(launcher, *args, **kwargs):
    """Run a @flyc.jit launcher and echo its GPU printf output into the notebook."""
    kwargs.setdefault("stream", torch.cuda.Stream())
    with pipes() as (out, _err):
        launcher(*args, **kwargs)
        torch.cuda.synchronize()
    print(out.read(), end="")

## 2. Two decorators: `@flyc.kernel` and `@flyc.jit`

FlyDSL splits a launch into two traced functions:

- **`@flyc.kernel`** marks **device** code — the body that runs on each GPU thread.
  Inside it you have intrinsics like `fx.thread_idx.x` and `fx.block_idx.x`.
- **`@flyc.jit`** marks the **host launcher**. It calls a kernel and `.launch(...)`es
  it with a grid/block configuration.

Both are *traced*, not interpreted: when first called, FlyDSL runs the Python once
to build MLIR, compiles it, and caches the result. So `block=(4, 1, 1)` is read at
**trace time**, while `fx.thread_idx.x` is a **runtime** value that differs per
thread.

Here is the smallest possible kernel — it takes no tensors and just prints.
`fx.printf` uses `{}` placeholders (avoid a literal `%` in the format string — it
is consumed by the underlying device `printf`).

In [ ]:
@flyc.kernel
def hello_kernel():
    bid = fx.block_idx.x
    tid = fx.thread_idx.x
    fx.printf("hello from block {} thread {}", bid, tid)


@flyc.jit
def hello(stream: fx.Stream = fx.Stream(None)):
    hello_kernel().launch(grid=(1, 1, 1), block=(4, 1, 1), stream=stream)


show_gpu_output(hello)

Four threads, four lines. The launch built a one-block grid of four threads, and
each thread reached the `printf`.

## 3. Looking at the generated IR

The fastest way to build intuition for what FlyDSL *did* is to read the MLIR it
produced. Set `FLYDSL_DUMP_IR=1` (and a dump directory) and FlyDSL writes one
`.mlir` file per compiler pass, from `00_origin.mlir` (the high-level `fly` IR)
down to the final ISA. The env var is read at **compile time**, so we set it, then
compile a fresh kernel and read back its first dump.

In [ ]:
import contextlib
import glob
import io
import os
import tempfile

dump_dir = tempfile.mkdtemp(prefix="flydsl_ir_")
os.environ["FLYDSL_DUMP_IR"] = "1"
os.environ["FLYDSL_DUMP_DIR"] = dump_dir


@flyc.kernel
def add_one_kernel(x: fx.Int32):
    fx.printf("x + 1 = {}", x + fx.Int32(1))


@flyc.jit
def add_one(x: fx.Int32, stream: fx.Stream = fx.Stream(None)):
    add_one_kernel(x).launch(grid=(1, 1, 1), block=(1, 1, 1), stream=stream)


# Compile + run once (silence the verbose per-pass dump log).
with contextlib.redirect_stdout(io.StringIO()):
    add_one(fx.Int32(41), stream=torch.cuda.Stream())
    torch.cuda.synchronize()

os.environ.pop("FLYDSL_DUMP_IR", None)   # stop dumping...
os.environ.pop("FLYDSL_DUMP_DIR", None)  # ...and clear the dump dir we set

origin = sorted(glob.glob(os.path.join(dump_dir, "*", "00_origin.mlir")))[0]
with open(origin) as f:
    print(f.read())

Things to notice in that high-level `fly` IR:

- `gpu.func @add_one_kernel_0(...) kernel { ... }` — the device kernel.
- `arith.addi %arg0, %c1_i32` — the `x + 1` you wrote, in MLIR form.
- `fly.print(...) {format = "..."}` — your `fx.printf`.
- `gpu.launch_func ... blocks in (...) threads in (...)` — the host-side launch.

The numbered files after `00_origin.mlir` show each lowering step (layout
lowering, `fly`→`rocdl`, `gpu`→`llvm`, …) down to `*_final_isa.s`. Whenever a
kernel misbehaves, dumping the IR is the first move.

---
**Next:** [`01_numeric_types`](01_numeric_types.ipynb) — the scalar type system you
just used (`fx.Int32`) in full: integers, floats, `bf16`, `fp8`, casts, and the
difference between compile-time and runtime values.